# 29 — Learned N-Gram Language Models

**Master syllabus coverage:** Added foundation between V2 Parts IV and V

## Why this matters

An n-gram model is the simplest learned next-token model. It exposes counting, smoothing, likelihood, generation, and context limits before neural-network complexity arrives.

## Learning objectives

- Estimate bigram counts and conditional probabilities from documents.
- Use smoothing to handle unseen transitions.
- Evaluate mean NLL and perplexity on complete sequences.
- Generate autoregressively and state the model's context limitation.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Learn a bigram table from data

A bigram model assumes the next token depends only on the current token:
$p(x_t\mid x_{<t})\approx p(x_t\mid x_{t-1})$. Count observed pairs, add smoothing,
and normalize each row. Unlike notebook 00, these parameters are estimated from text.


In [ ]:
import numpy as np

corpus = ["timing is funny", "timing is everything", "funny timing wins", "timing wins"]
words = sorted({word for line in corpus for word in line.split()})
vocab = ["<BOS>", "<EOS>", *words]
stoi = {token: index for index, token in enumerate(vocab)}
itos = dict(enumerate(vocab))
V = len(vocab)
counts = np.zeros((V, V), dtype=np.int64)

for line in corpus:
    sequence = ["<BOS>", *line.split(), "<EOS>"]
    for left, right in zip(sequence, sequence[1:]):
        counts[stoi[left], stoi[right]] += 1
print("vocabulary:", vocab)
print("observed transitions:", int((counts > 0).sum()))


## 2. Smoothing reserves probability for unseen pairs

Maximum-likelihood rows are count divided by row total, but unseen events get zero
probability and therefore infinite evaluation loss. Add-$\alpha$ smoothing uses
$(c_{ij}+\alpha)/(\sum_j c_{ij}+\alpha V)$. It is simple, not state of the art.


In [ ]:
def bigram_probabilities(count_matrix: np.ndarray, alpha: float) -> np.ndarray:
    smoothed = count_matrix.astype(np.float64) + alpha
    return smoothed / smoothed.sum(axis=1, keepdims=True)

probabilities = bigram_probabilities(counts, alpha=0.1)
np.testing.assert_allclose(probabilities.sum(axis=1), 1.0)
timing_row = probabilities[stoi["timing"]]
print("p(next | timing):", {itos[i]: round(p, 3) for i, p in enumerate(timing_row) if p > 0.02})


## 3. Evaluate held-out negative log-likelihood

Token NLL measures every predicted transition. Perplexity is its exponent. Split source
documents before extracting n-grams; otherwise near-identical local transitions leak
across splits. An n-gram baseline reveals whether a neural model actually learns longer
dependencies rather than only local frequency.


In [ ]:
def sequence_nll(text: str, table: np.ndarray) -> float:
    sequence = ["<BOS>", *text.split(), "<EOS>"]
    log_probabilities = []
    for left, right in zip(sequence, sequence[1:]):
        if left not in stoi or right not in stoi:
            raise ValueError("held-out text contains out-of-vocabulary word")
        log_probabilities.append(np.log(table[stoi[left], stoi[right]]))
    return float(-np.mean(log_probabilities))

for text in ("timing is funny", "funny timing wins", "everything timing is"):
    nll = sequence_nll(text, probabilities)
    print(f"{text!r}: NLL={nll:.3f}, PPL={np.exp(nll):.2f}")


## 4. Autoregressive generation

Generation uses the same row repeatedly, appending the sampled token until EOS or a
length cap. A bigram cannot distinguish two contexts ending in the same token; this is
its Markov limitation and the motivation for richer context representations.


In [ ]:
rng = np.random.default_rng(7)

def generate(max_tokens: int = 12) -> str:
    current = "<BOS>"
    output = []
    for _ in range(max_tokens):
        next_id = int(rng.choice(V, p=probabilities[stoi[current]]))
        current = itos[next_id]
        if current == "<EOS>":
            break
        if current != "<BOS>":
            output.append(current)
    return " ".join(output)

for _ in range(8):
    print(generate())


## Failure modes to recognize

- **Zero-probability event:** one unseen transition makes sequence NLL infinite.
- **Row with no observations:** unsmoothed normalization divides by zero.
- **Document leakage:** transitions from one source appear in both train and evaluation.
- **Context overclaim:** a bigram is credited with dependencies it cannot represent.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Implement a trigram model and define its fallback for an unseen two-token context.
2. Sweep smoothing alpha and report train versus held-out NLL.
3. Compare generated samples with shuffled unigram samples and explain the difference.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can learn, smooth, evaluate, and sample an n-gram model while describing exactly what context it forgets.

**Next:** 30 — BPE, unigram tokenization, and vocabulary design.
